# Notebook 02: Community Summaries
## Turning Graph Clusters into Natural-Language Reports

We now have a graph that has been partitioned into **15 communities** (C1 level). Each
community is a cluster of closely-related entities — the 2022 finalists, the host-nation
infrastructure, the FIFA governance controversies, and so on.

In this notebook we ask the LLM to read the entities and relationships of each community
and write a structured **JSON report**. These reports are the bridge between the raw graph
and the final query: in Notebook 03 they feed the Map-Reduce step that answers global
sensemaking questions.

**Paper reference:** Edge et al. (2025) — §3.1.5 *Community Summarization*, Appendix E.2.

### What this notebook does

```
wc2022_trimmed_graph.json  +  wc2022_trimmed_communities.json
           │
           ▼  For each C1 community:
           │    1. Collect entity descriptions + intra-community edge descriptions
           │    2. Prioritise by edge weight (§ 3.1.5)
           │    3. LLM → structured JSON report  (Appendix E.2)
           ▼
data/output/wc2022_trimmed_summaries.json
```

## Imports and Setup

In [ ]:
import json
import re
import subprocess
from pathlib import Path

import networkx as nx
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from IPython.display import Markdown, display

Path("../data/output").mkdir(parents=True, exist_ok=True)
print("Imports OK")

In [ ]:
PDF_STEM = "wc2022_trimmed"

GRAPH_PATH       = Path("../data/output") / f"{PDF_STEM}_graph.json"
COMMUNITIES_PATH = Path("../data/output") / f"{PDF_STEM}_communities.json"
SUMMARIES_PATH   = Path("../data/output") / f"{PDF_STEM}_summaries.json"

print(f"Graph      : {GRAPH_PATH}")
print(f"Communities: {COMMUNITIES_PATH}")
print(f"Output     : {SUMMARIES_PATH}")

### LLM Setup

Same portable Ollama configuration as Notebook 00 (env var → WSL2 → localhost). We increase
`num_predict` to **1500** because community reports include a `findings` array with multiple
entries — they are longer than the extraction tuples.

In [ ]:
import os
import platform


def ollama_base_url() -> str:
    """Locate the Ollama server so this notebook runs on any machine.

    Priority: OLLAMA_BASE_URL env var  →  WSL2 Windows-host IP  →  localhost.
    Set OLLAMA_BASE_URL yourself if Ollama runs somewhere else (e.g. a remote box).
    """
    if os.environ.get("OLLAMA_BASE_URL"):
        return os.environ["OLLAMA_BASE_URL"]
    if "microsoft" in platform.uname().release.lower():   # WSL2 → Ollama on the Windows host
        host = subprocess.check_output(
            "ip route list default | awk '{print $3}'", shell=True
        ).decode().strip()
        if host:
            return f"http://{host}:11434"
    return "http://localhost:11434"                        # native Linux / macOS / Windows

MODEL    = "qwen2.5:7b"
BASE_URL = ollama_base_url()

llm = ChatOllama(
    model=MODEL,
    base_url=BASE_URL,
    temperature=0.0,
    num_predict=1500,
)

print(f"LLM: {MODEL}  @  {BASE_URL}")

---
## Step 1 — Load Graph and Communities  *(§ 3.1.5)*

In [ ]:
with open(GRAPH_PATH, encoding="utf-8") as f:
    G = nx.node_link_graph(json.load(f))

with open(COMMUNITIES_PATH, encoding="utf-8") as f:
    communities = json.load(f)

c1 = communities["C1"]
print(f"Graph      : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"C1 communities: {len(c1)}")
for cid, members in c1.items():
    intra = sum(1 for u, v in G.edges(members) if v in members)
    print(f"  [{cid}] {len(members):>3} nodes, {intra:>3} intra-edges — {members[:3]}")

---
## Step 2 — Build Community Context  *(§ 3.1.5)*

For each community we assemble two lists:

1. **Entity descriptions** — one line per node: `NAME (TYPE): description`  
   Sorted by degree descending so the most central entities appear first in the prompt.

2. **Relationship descriptions** — one line per intra-community edge: `SRC → TGT: description`  
   Sorted by `degree(src) + degree(tgt)` descending, as the paper specifies (§3.1.5).
   This ensures the most important relationships fill the context window first.

In [ ]:
def build_community_context(G: nx.Graph, members: list[str]) -> tuple[str, str]:
    member_set = set(members)

    # Entity descriptions — sorted by degree (descending)
    nodes_sorted = sorted(
        [n for n in members if n in G.nodes],
        key=lambda n: G.degree(n),
        reverse=True,
    )
    node_lines = [
        f"{n} ({G.nodes[n].get('type', 'UNKNOWN')}): {G.nodes[n].get('description', '')}"
        for n in nodes_sorted
    ]

    # Intra-community edge descriptions — sorted by combined degree (descending)
    intra_edges = [
        (u, v, d)
        for u, v, d in G.edges(members, data=True)
        if u in member_set and v in member_set
    ]
    intra_edges.sort(key=lambda e: G.degree(e[0]) + G.degree(e[1]), reverse=True)
    edge_lines = [
        f"{u} → {v}: {d.get('description', '')}  (weight: {d.get('weight', 1)})"
        for u, v, d in intra_edges
    ]

    return "\n".join(node_lines), "\n".join(edge_lines)


# Quick preview for community [2] (Argentina / France / Croatia / Morocco)
sample_nodes, sample_edges = build_community_context(G, c1["2"])
print("=== Nodes (first 3) ===")
print("\n".join(sample_nodes.split("\n")[:3]))
print("\n=== Edges (first 3) ===")
print("\n".join(sample_edges.split("\n")[:3]))

---
## Step 3 — The Community Summary Prompt  *(Appendix E.2)*

The paper's Appendix E.2 specifies a structured JSON output with five fields. We follow it
verbatim. The `rating` (0–10 float) is particularly important for Notebook 03: when multiple
community summaries are retrieved during the Map step, helpfulness scores determine which
ones feed the final Reduce step.

```
---Goal---
Write a comprehensive report of a community, given a list of entities that
belong to the community and their relationships.

---Report Structure---
Return ONLY a JSON object with exactly these fields:
{
  "title":              "<short, specific title naming key entities>",
  "summary":            "<executive summary, 2–4 sentences>",
  "rating":             <float 0–10, overall importance of this community>,
  "rating_explanation": "<one sentence explaining the rating>",
  "findings": [
    {"summary": "<insight title>", "explanation": "<2–3 sentences>"},
    ...  (3–5 findings)
  ]
}

---Community Data---
Entities:
{node_descriptions}

Relationships:
{edge_descriptions}

Output:
```

In [ ]:
SUMMARY_PROMPT = """\
---Goal---
Write a comprehensive report of a community, given a list of entities that belong to the
community and their relationships.

---Report Structure---
Return ONLY a JSON object with exactly these fields:
{{
  "title":              "<short, specific title naming key entities>",
  "summary":            "<executive summary, 2-4 sentences>",
  "rating":             <float 0-10, overall importance of this community>,
  "rating_explanation": "<one sentence explaining the rating>",
  "findings": [
    {{"summary": "<insight title>", "explanation": "<2-3 sentences>"}},
    ... (3-5 findings total)
  ]
}}

---Community Data---
Entities:
{node_descriptions}

Relationships:
{edge_descriptions}

Output:"""


def extract_json(text: str) -> dict:
    """Extract the first JSON object from LLM output, handling markdown fences."""
    # Strip markdown code fences if present
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    # Find the outermost { ... }
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        return json.loads(m.group(0))
    raise ValueError(f"No JSON object found in LLM output: {text[:300]}")


def summarise_community(G: nx.Graph, members: list[str]) -> dict:
    node_desc, edge_desc = build_community_context(G, members)
    prompt = SUMMARY_PROMPT.format(
        node_descriptions=node_desc,
        edge_descriptions=edge_desc,
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return extract_json(response.content)


print("Prompt and helpers defined.")

---
## Step 4 — Generate Summaries for All C1 Communities

We iterate over all 15 C1 communities. Expect ~1–2 minutes total on `qwen2.5:7b`.

Each result is appended to `summaries` immediately, so a partial run is not lost if the
kernel is interrupted.

In [ ]:
summaries = []

for cid, members in c1.items():
    print(f"Community [{cid}]  ({len(members)} nodes) ...", end=" ", flush=True)
    try:
        report = summarise_community(G, members)
        report["community_id"] = cid
        report["member_count"] = len(members)
        summaries.append(report)
        print(f"OK — {report.get('title', '?')[:60]}")
    except Exception as e:
        print(f"FAILED: {e}")
        summaries.append({
            "community_id": cid,
            "member_count": len(members),
            "title": f"Community {cid} (parse error)",
            "summary": "",
            "rating": 0.0,
            "rating_explanation": "",
            "findings": [],
            "_error": str(e),
        })

print(f"\nDone. {sum(1 for s in summaries if '_error' not in s)}/{len(summaries)} succeeded.")

---
## Step 5 — Save `summaries.json`

In [ ]:
# Sort by rating descending so the most important communities appear first
summaries.sort(key=lambda s: s.get("rating", 0), reverse=True)

with open(SUMMARIES_PATH, "w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)

print(f"Saved {SUMMARIES_PATH}  ({SUMMARIES_PATH.stat().st_size / 1024:.1f} KB)")
print(f"\nRatings (community_id — rating — title):")
for s in summaries:
    print(f"  [{s['community_id']}] {s.get('rating', 0):>4.1f}  {s.get('title', '?')[:60]}")

---
## Inspect — Read the Discovered Themes

Below we render each community report as formatted Markdown. This is the knowledge that
the Map-Reduce query in Notebook 03 will draw upon.

In [ ]:
for s in summaries:
    if "_error" in s:
        continue
    lines = [
        f"### [{s['community_id']}] {s.get('title', 'Untitled')}",
        f"**Rating:** {s.get('rating', '?')}/10 — {s.get('rating_explanation', '')}",
        f"",
        s.get('summary', ''),
        "",
    ]
    for finding in s.get('findings', []):
        lines.append(f"**{finding.get('summary', '')}**")
        lines.append(finding.get('explanation', ''))
        lines.append("")
    lines.append("---")
    display(Markdown("\n".join(lines)))

---
## Summary

| Paper section | What we did |
|---|---|
| § 3.1.5 | Collected node + intra-edge descriptions per community |
| § 3.1.5 | Prioritised edges by combined source+target degree |
| App. E.2 | Used paper's exact JSON output format (title, summary, rating, findings) |

The summaries are saved at `data/output/wc2022_trimmed_summaries.json`, sorted by rating.

**Next → Notebook 03:** We use these summaries in a **Map-Reduce** query. For the question
*"What are the main storylines and themes of the 2022 FIFA World Cup?"*, each summary is
scored for helpfulness (Map), then the top-scoring ones are synthesised into a final answer
(Reduce) — § 3.1.6, Appendix E.3–E.4.